## Dataset Assembly — EdNet KT1

This notebook loads and assembles a sample of the **EdNet KT1** dataset, which contains per-student CSV files recording question-answering interactions (timestamps, question IDs, answers, and elapsed times). Because the full dataset contains ~725,000 files, a limit is applied so that only a manageable subset is loaded for exploration and experimentation.

---

### 1 — Import Libraries

In [1]:
import pandas as pd   # DataFrame manipulation and CSV I/O
import glob           # Pattern-based file discovery (e.g. *.csv)
import os             # OS-level path utilities (basename, join, etc.)
import bramlib        # Custom helper library (e.g. analyze_missing_data)

---

### 2 — Helper Functions

Three reusable functions are defined here to keep the assembly logic clean and DRY.

#### `create_df(file, label=None)`
Reads a **single** per-student CSV file and:
1. Extracts the numeric **user ID** from the filename (e.g. `u12345.csv` → `12345`).
2. Adds that ID as a new `user_id` column so rows can be traced back to a student after merging.
3. Optionally **prefixes every non-key column** with a `label` string. This is useful when joining data from different feature folders (e.g. `lecture_`, `review_`) so column names remain unambiguous after the merge.

#### `combine_folder(folder, limit=-1)`
Reads **all CSV files** in a given folder up to an optional `limit`. It calls `create_df` for each file, appends the result to a list, and concatenates everything into one large DataFrame. Progress is printed every 100 files to monitor long-running loads.

> **Why a limit?** The raw KT1 folder contains ~725,000 files. Loading all of them would produce a ~20 GB DataFrame. The limit lets you work with a representative slice (e.g. 25,000 files ≈ 6 M rows, ~282 MB) during development.

#### `get_other_folders(folders, main_df)`
Enriches the main DataFrame with **supplementary feature columns** from other data folders (e.g. lecture interactions, review actions). For each extra folder it:
1. Discovers all CSV files and identifies which users are **already present** in `main_df`.
2. Loads only the matching files (to avoid pulling in users not in the main set).
3. Merges the new columns into `main_df` on the shared keys `user_id` and `timestamp` using an **outer join** so no rows are accidentally dropped.
4. Prints the count of **overlapping timestamps** as a quick sanity-check that the join found real matches.

In [2]:
def create_df(file, label=None):
    """Read one per-student CSV and attach the user ID derived from the filename."""
    df = pd.read_csv(file)

    # Filenames are formatted as 'u<id>.csv'; strip the leading 'u' and cast to int.
    user_id = os.path.basename(file).split(".")[0]
    user_id = int(user_id[1:])
    df["user_id"] = user_id

    # When loading supplementary folders, prefix column names with the folder label
    # (e.g. 'elapsed_time' becomes 'lecture_elapsed_time') so columns stay unique
    # after merging multiple feature sources.
    if label:
        df.columns = [f"{label}_{col}" if col not in ["timestamp", "user_id"] else col for col in df.columns]
    return df


def combine_folder(folder, limit=-1):
    """Concatenate all CSVs in a folder into a single DataFrame, with an optional row limit."""
    all_files = glob.glob(os.path.join(folder, "*.csv"))  # Discover every CSV in the folder
    df_list = []

    n = 0
    for file in all_files[:limit]:  # Slice the file list to respect the limit
        df = create_df(file=file)
        df_list.append(df)

        n += 1
        if n % 100 == 0:
            print(f"Currently on:{n}")  # Progress indicator every 100 files

    combined_df = pd.concat(df_list, ignore_index=True)  # Stack all per-student DataFrames
    return combined_df


def get_other_folders(folders, main_df):
    """Merge supplementary feature columns from additional data folders into the main DataFrame."""
    for folder in folders:
        # Use the folder name as the column-prefix label (lowercased for consistency).
        folder_label = os.path.basename(os.path.normcase(folder)).lower()
        print(f"currently on: {folder_label}")

        extra_dfs = []
        files = glob.glob(os.path.join(folder, "*.csv"))

        # Only load files for users that already exist in main_df;
        # this avoids inflating the DataFrame with unknown users.
        valid_users = main_df["user_id"].unique()

        for file in files:
            userc = os.path.basename(file).split(".")[0]
            userc_id = int(userc[1:])

            if userc_id in valid_users:
                df_extra = create_df(file, label=folder_label)
                extra_dfs.append(df_extra)

        if extra_dfs:
            temp_df = pd.concat(extra_dfs, ignore_index=True)

            # Outer join on the two shared keys so no rows are silently dropped.
            main_df = pd.merge(main_df, temp_df, on=["user_id", "timestamp"], how="outer")

            # Quick sanity-check: how many timestamps actually matched between the two sets?
            common = set(main_df["timestamp"]).intersection(set(temp_df["timestamp"]))
            print(len(common))
    return main_df

---

### 3 — Load the KT1 Data

`combine_folder` is called with `limit=25000`, which loads the first 25,000 student files. This produces roughly **6.17 million rows** and occupies ~282 MB of RAM — large enough to be representative, but small enough to remain workable on a standard laptop.

`df.head(10)` is used immediately after loading to do a quick visual sanity-check: confirm the expected columns are present, values look reasonable, and the `user_id` column was attached correctly.

In [4]:
df = combine_folder(r"C:\Users\bramm\S3C2-Data\Student grade project\Data\raw\EdNet-KT1\KT1", 25000)
df.head(10)

Currently on:100
Currently on:200
Currently on:300
Currently on:400
Currently on:500
Currently on:600
Currently on:700
Currently on:800
Currently on:900
Currently on:1000
Currently on:1100
Currently on:1200
Currently on:1300
Currently on:1400
Currently on:1500
Currently on:1600
Currently on:1700
Currently on:1800
Currently on:1900
Currently on:2000
Currently on:2100
Currently on:2200
Currently on:2300
Currently on:2400
Currently on:2500
Currently on:2600
Currently on:2700
Currently on:2800
Currently on:2900
Currently on:3000
Currently on:3100
Currently on:3200
Currently on:3300
Currently on:3400
Currently on:3500
Currently on:3600
Currently on:3700
Currently on:3800
Currently on:3900
Currently on:4000
Currently on:4100
Currently on:4200
Currently on:4300
Currently on:4400
Currently on:4500
Currently on:4600
Currently on:4700
Currently on:4800
Currently on:4900
Currently on:5000
Currently on:5100
Currently on:5200
Currently on:5300
Currently on:5400
Currently on:5500
Currently on:5600
C

,timestamp,solving_id,question_id,user_answer,elapsed_time,user_id
0,1565096190868,1,q5012,b,38000,1
1,1565096221062,2,q4706,c,24000,1
2,1565096293432,3,q4366,b,68000,1
3,1565096339668,4,q4829,a,42000,1
4,1565096401774,5,q6528,b,59000,1
5,1565096463370,6,q4793,a,58000,1
6,1565096501746,7,q6488,a,35000,1
7,1565097101361,8,q356,b,23000,1
8,1565097171393,9,q1382,c,22000,1
9,1565097240758,10,q830,b,25000,1


---

### 4 — Inspect Data Types and Memory Usage

`df.info()` prints a concise summary of the DataFrame:

| What it tells you | Why it matters |
|---|---|
| **Row / column count** | Confirms the expected volume was loaded (6.17 M rows, 6 columns). |
| **Dtype per column** | `int64` columns support numeric operations (mean, std, range); `object` columns are strings and need encoding or mapping before modelling. |
| **Memory usage** | Helps anticipate RAM pressure. ~282 MB is comfortably in-memory, but scaling to the full dataset would require chunking or a database. |

Knowing that `question_id` and `user_answer` are `object` (string) columns is especially important: they will need to be label-encoded or one-hot-encoded before being fed to most ML models.

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6171161 entries, 0 to 6171160
Data columns (total 6 columns):
 #   Column        Dtype 
---  ------        ----- 
 0   timestamp     int64 
 1   solving_id    int64 
 2   question_id   object
 3   user_answer   object
 4   elapsed_time  int64 
 5   user_id       int64 
dtypes: int64(4), object(2)
memory usage: 282.5+ MB


---

### 5 — Missing Data Report

`bramlib.analyze_missing_data(df)` is a custom utility that produces:
- A **printed summary** (total data points, total missing values, overall missing %).
- A **bar chart** visualising the missing-value rate per column.

The output shows that only **854 out of 37 million** data points are missing (0.00%), which is negligible. This means no imputation strategy is required at this stage. It's still good practice to confirm this explicitly before modelling, because even a tiny fraction of nulls can break certain algorithms if left unhandled.

In [6]:
bramlib.analyze_missing_data(df)

--- Missing Data Report for df ---
Total Data Points: 37026966
Total Missing Data Points: 854
Overall Missing Data: 0.00%

Missing Data Per Column (%):


---

### 6 — Export the Assembled Dataset

The assembled DataFrame is written to a CSV file so it can be reused in downstream notebooks or shared with collaborators without having to re-run the expensive loading step.

> **Note:** `pd.DataFrame.to_csv()` writes the pandas row-index as the first column by default. If you want to omit it, pass `index=False`.

In [7]:
df.to_csv("25kfiles_kt1.csv")